# Telco Customer Churn Analysis — 500 Customers

This notebook performs data cleaning, duplicate handling, missing-value treatment, IQR outlier analysis, exploratory analysis, 8 visualizations, and business insights.

In [1]:
# Telco Customer Churn Analysis — 500 Customers

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load raw data
df = pd.read_csv("telco_500_raw.csv")
df.head()

# Basic information
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)

# Missing values
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

# Duplicate records
print("Duplicate rows:", df.duplicated().sum())

# Clean missing values
clean = df.copy()
for col in clean.columns:
    if clean[col].dtype == "object":
        clean[col] = clean[col].fillna(clean[col].mode()[0])
    else:
        clean[col] = clean[col].fillna(clean[col].median())

# Remove duplicates
clean = clean.drop_duplicates().reset_index(drop=True)

# Data types
display(clean.dtypes)

# IQR outlier analysis
numeric_cols = ["TenureMonths", "MonthlyCharges", "TotalCharges"]
for col in numeric_cols:
    q1 = clean[col].quantile(0.25)
    q3 = clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((clean[col] < lower) | (clean[col] > upper)).sum()
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
          f"Lower={lower:.2f}, Upper={upper:.2f}, Outliers={count}")

# Churn analysis
churn_rate = (clean["Churn"].eq("Yes").mean()) * 100
print(f"Overall churn rate: {churn_rate:.2f}%")

display(clean.groupby("Contract")["Churn"].apply(lambda x: (x=="Yes").mean()*100).sort_values(ascending=False))
display(clean.groupby("InternetService")["Churn"].apply(lambda x: (x=="Yes").mean()*100).sort_values(ascending=False))
display(clean.groupby("PaymentMethod")["Churn"].apply(lambda x: (x=="Yes").mean()*100).sort_values(ascending=False))

# Visualization 1
clean["Churn"].value_counts().plot(kind="bar", title="1. Customer Churn Distribution")
plt.xlabel("Churn"); plt.ylabel("Customers"); plt.show()

# Visualization 2
pd.crosstab(clean["Contract"], clean["Churn"], normalize="index").mul(100).plot(
    kind="bar", title="2. Churn Rate by Contract Type")
plt.ylabel("Percentage"); plt.xticks(rotation=0); plt.show()

# Visualization 3
clean["MonthlyCharges"].plot(kind="hist", bins=25, title="3. Monthly Charges Distribution")
plt.xlabel("Monthly Charges"); plt.show()

# Visualization 4
clean["TenureMonths"].plot(kind="hist", bins=24, title="4. Tenure Distribution")
plt.xlabel("Tenure (Months)"); plt.show()

# Visualization 5
pd.crosstab(clean["InternetService"], clean["Churn"], normalize="index").mul(100).plot(
    kind="bar", title="5. Churn Rate by Internet Service")
plt.ylabel("Percentage"); plt.xticks(rotation=0); plt.show()

# Visualization 6
clean.groupby("PaymentMethod")["Churn"].apply(lambda x: (x=="Yes").mean()*100).sort_values().plot(
    kind="bar", title="6. Churn Rate by Payment Method")
plt.ylabel("Churn Rate (%)"); plt.xticks(rotation=30, ha="right"); plt.show()

# Visualization 7
clean.boxplot(column="MonthlyCharges", by="Churn")
plt.title("7. Monthly Charges by Churn Status"); plt.suptitle("")
plt.xlabel("Churn"); plt.ylabel("Monthly Charges"); plt.show()

# Visualization 8
clean.boxplot(column="TenureMonths", by="Churn")
plt.title("8. Tenure by Churn Status"); plt.suptitle("")
plt.xlabel("Churn"); plt.ylabel("Tenure (Months)"); plt.show()

# Save cleaned data
clean.to_csv("telco_500_cleaned.csv", index=False)

print("Final cleaned customers:", len(clean))
print("Analysis completed successfully.")


Matplotlib is building the font cache; this may take a moment.


Shape: (505, 20)
<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        505 non-null    str    
 1   Gender            505 non-null    str    
 2   SeniorCitizen     505 non-null    str    
 3   Partner           505 non-null    str    
 4   Dependents        505 non-null    str    
 5   TenureMonths      505 non-null    int64  
 6   PhoneService      505 non-null    str    
 7   InternetService   505 non-null    str    
 8   OnlineSecurity    497 non-null    str    
 9   OnlineBackup      505 non-null    str    
 10  DeviceProtection  505 non-null    str    
 11  TechSupport       497 non-null    str    
 12  StreamingTV       505 non-null    str    
 13  StreamingMovies   505 non-null    str    
 14  Contract          505 non-null    str    
 15  PaperlessBilling  505 non-null    str    
 16  PaymentMethod     505 non-null    str 

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,505,500,CUST0011,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,505,2,Male,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,505,2,No,412,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Partner,505,2,No,255,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,505,2,No,361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TenureMonths,505.0,NaN,NaN,NaN,36.520792,20.8499,1.0,18.0,37.0,53.0,72.0
PhoneService,505,2,Yes,458,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,505,3,Fiber optic,243,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,497,3,No,284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineBackup,505,3,No,246,NaN,NaN,NaN,NaN,NaN,NaN,NaN


TotalCharges      8
MonthlyCharges    8
OnlineSecurity    8
TechSupport       8
dtype: int64

Duplicate rows: 5


TypeError: Cannot perform reduction 'median' with string dtype